In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input

from tensorflow.keras.callbacks import Callback, EarlyStopping

## Simulation1

In [ ]:
X_id = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Target1.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)
X_allog = pd.read_excel('Targetx.xlsx', index_col=0,sheet_name="allog").values.astype(np.float32)
Y = pd.read_excel('orthograph_sim1_2.xlsx').values[0:477,0:350].astype(np.float32)

# 1) Merge the three feature blocks into one matrix
X_all = np.concatenate([X_id, X_pos, X_allog], axis=1)

# 2) Single-input model
inp = Input(shape=(X_all.shape[1],), name='in_all')
h1  = Dense(700, activation='sigmoid')(inp)
h2  = Dense(500, activation='sigmoid')(h1)
out = Dense(350, activation='sigmoid')(h2)  #

model_merged = Model(inputs=inp, outputs=out)

# 3) Compile & train
model_merged.compile(optimizer="adam", loss="mse")
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)

model_merged.fit(X_all, Y, batch_size=25, epochs=700, callbacks=[callback])

model_merged.save('simulation_1_model_merged.h5')

Epoch 1/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.1124
Epoch 2/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0182
Epoch 3/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0181
Epoch 4/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0180
Epoch 5/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0177
Epoch 6/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0176
Epoch 7/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0176
Epoch 8/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0176
Epoch 9/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0172
Epoch 10/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0174
Epoch 11/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0168
Epoch 12/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0162
Epoch 13/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0158
Epoch 14/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0158
Epoch 15/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - lo

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
data_lst = ["TL+Allog1.xlsx", "TL-Allog1.xlsx", "SubL+Allog1.xlsx", "SubL-Allog1.xlsx"]

# orthographic vectors
Y = pd.read_excel('orthograph_sim1_2.xlsx').values[0:477,0:350].astype(np.float32)

# output DataFrame and a dict to hold preds
df_sim1 = pd.DataFrame()
preds = {}


for doc in data_lst:

    key = doc.replace('1.xlsx','').lower()
    X_id    = pd.read_excel(doc, sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos   = pd.read_excel(doc, sheet_name='position', index_col=0).values.astype(np.float32)
    allog_f = doc.replace('1.xlsx','x.xlsx')
    X_allog   = pd.read_excel(allog_f, sheet_name='allog', index_col=0).values.astype(np.float32)
    X_all = np.concatenate([X_id, X_pos, X_allog], axis=1).astype(np.float32)
    pred = model_merged.predict(X_all)
    preds[key] = pred

    # per-word correlation distances
    valz = []
    for i in range(len(Y)):
        r = np.corrcoef(pred[i], Y[i])[0,1]
        valz.append(round(1-r,3))

    df_sim1[f'target-{key}'] = valz


pairs = [('tl-allog',  'subl-allog'),('tl+allog', 'subl+allog'),]
for a, b in pairs:
    df_sim1[f'{a}-{b}'] = [
        round(1 - np.corrcoef(preds[a][i], preds[b][i])[0,1], 4)
        for i in range(len(Y))]


df_sim1.to_excel('simulation_1c_merged_model_correlation_test_1.xlsx')


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


# Correlation Distance Between words that share the same root

In [ ]:
import itertools

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
X_all = np.concatenate([X_id, X_pos, X_allog], axis=1).astype(np.float32)
pred_all = model_merged.predict(X_all)


df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


# Load roots file
df_roots = pd.read_excel('targets with roots and semantics.xlsx', sheet_name='grouped by roots',     index_col=0)


# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')


# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('Root'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(pred_all.shape[1])]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'Root': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('Root')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()


with pd.ExcelWriter('simulation_1c_merged_model_root_pairwise_distances_1.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_root_summary', index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


# Correlation Distance Between the word and other 4 conditions

In [ ]:
import numpy as np
import pandas as pd

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
X_all = np.concatenate([X_id, X_pos, X_allog], axis=1).astype(np.float32)
pred_all = model_merged.predict(X_all)


df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words



PAIRS_XLSX  = 'targets with roots and semantics.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)

# keep exactly 4 columns in this order
out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation_1c_merged_model_4_conditions_pairs_with_distance_1.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
